# 03 — Urbanization
**Source:** India Districts Census 2011 · 640 districts  
Covers urban/rural household splits, urbanization %, and digital/energy access as development proxies.

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.households,
           f.rural_households, f.urban_households,
           f.lpg_or_png_households,
           f.housholds_with_electric_lighting,
           f.households_with_internet,
           f.households_with_computer,
           f.households_with_telephone_mobile_phone
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

df = df[df['households'] > 0]
df['urban_pct']    = (df['urban_households'] / df['households'] * 100).round(2)
df['rural_pct']    = (df['rural_households'] / df['households'] * 100).round(2)
df['electricity_pct'] = (df['housholds_with_electric_lighting'] / df['households'] * 100).round(2)
df['lpg_pct']      = (df['lpg_or_png_households'] / df['households'] * 100).round(2)
df['internet_pct'] = (df['households_with_internet'] / df['households'] * 100).round(2)
df['computer_pct'] = (df['households_with_computer'] / df['households'] * 100).round(2)
df['mobile_pct']   = (df['households_with_telephone_mobile_phone'] / df['households'] * 100).round(2)

state = df.groupby('state_name').agg(
    households=('households','sum'),
    rural_households=('rural_households','sum'),
    urban_households=('urban_households','sum'),
    lpg=('lpg_or_png_households','sum'),
    electricity=('housholds_with_electric_lighting','sum'),
    internet=('households_with_internet','sum'),
    computer=('households_with_computer','sum'),
    mobile=('households_with_telephone_mobile_phone','sum'),
).reset_index()

state['urban_pct']      = (state['urban_households'] / state['households'] * 100).round(2)
state['electricity_pct']= (state['electricity'] / state['households'] * 100).round(2)
state['lpg_pct']        = (state['lpg'] / state['households'] * 100).round(2)
state['internet_pct']   = (state['internet'] / state['households'] * 100).round(2)
state['computer_pct']   = (state['computer'] / state['households'] * 100).round(2)
state['mobile_pct']     = (state['mobile'] / state['households'] * 100).round(2)

national_urban = df['urban_households'].sum() / df['households'].sum() * 100
print(f'National Urbanization Rate: {national_urban:.2f}%')

## 3.1 — Urbanization Rate by State (Bar)

In [ ]:
state_sorted = state.sort_values('urban_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if u < 20 else '#f39c12' if u < 40 else '#3498db' if u < 60 else '#2ecc71'
          for u in state_sorted['urban_pct']]
bars = ax.barh(state_sorted['state_name'], state_sorted['urban_pct'], color=colors)
ax.axvline(national_urban, color='black', linestyle='--', label=f'National avg: {national_urban:.1f}%')
ax.set_xlabel('Urban Household %')
ax.set_title('Urbanization Rate by State\n(Red <20% | Orange 20–40% | Blue 40–60% | Green >60%)',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, state_sorted['urban_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()

## 3.2 — Rural vs Urban Household Split by State (100% Stacked)

In [ ]:
state_sorted2 = state.sort_values('urban_pct', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Urban', x=state_sorted2['state_name'], y=state_sorted2['urban_pct'],
                     marker_color='#3498db'))
fig.add_trace(go.Bar(name='Rural', x=state_sorted2['state_name'],
                     y=100 - state_sorted2['urban_pct'], marker_color='#27ae60'))

fig.update_layout(
    barmode='stack',
    title='Rural vs Urban Household Share by State (%) — sorted most urban to least',
    yaxis_title='% of Households',
    xaxis_tickangle=-45,
    height=500,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 3.3 — Top & Bottom 10 Districts by Urban Household %

In [ ]:
top10    = df.nlargest(10, 'urban_pct')[['district_name','state_name','urban_pct']]
bottom10 = df[df['urban_pct'] > 0].nsmallest(10, 'urban_pct')[['district_name','state_name','urban_pct']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top10['district_name'] + ', ' + top10['state_name'], top10['urban_pct'],
         color=sns.color_palette('Blues_r', 10))
ax1.set_xlabel('Urban HH %'); ax1.set_title('Top 10 Most Urban Districts', fontweight='bold')
ax1.invert_yaxis()

ax2.barh(bottom10['district_name'] + ', ' + bottom10['state_name'], bottom10['urban_pct'],
         color=sns.color_palette('Greens_r', 10))
ax2.set_xlabel('Urban HH %'); ax2.set_title('Top 10 Most Rural Districts', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout(); plt.show()

## 3.4 — Development Access Indicators by State (Heatmap)

In [ ]:
heat_data = state.set_index('state_name')[['electricity_pct','lpg_pct','internet_pct','computer_pct','mobile_pct']]
heat_data.columns = ['Electricity','LPG/PNG','Internet','Computer','Mobile Phone']
heat_data = heat_data.sort_values('Electricity', ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(heat_data, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=.5,
            cbar_kws={'label': '% of Households'}, ax=ax)
ax.set_title('Household Access to Key Development Amenities by State (%)\nsorted by Electricity Access',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 3.5 — Internet Access vs Urbanization (District Scatter)

In [ ]:
fig = px.scatter(
    df, x='urban_pct', y='internet_pct',
    color='state_name', hover_name='district_name',
    hover_data={'state_name':True,'urban_pct':':.1f','internet_pct':':.2f'},
    labels={'urban_pct':'Urban Household %','internet_pct':'Internet Access %'},
    title='Internet Access vs Urbanization Rate — District Level',
    trendline='ols', trendline_scope='overall', trendline_color_override='black'
)
fig.update_layout(showlegend=False, height=550)
fig.show()

## 3.6 — Urbanization Distribution (Histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['urban_pct'], bins=40, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(df['urban_pct'].mean(), color='red', linestyle='--', label=f"Mean: {df['urban_pct'].mean():.1f}%")
ax.axvline(50, color='green', linestyle='--', label='50% mark')
ax.set_xlabel('Urban Household %'); ax.set_ylabel('Number of Districts')
ax.set_title('Distribution of Urbanization Rate Across 640 Districts', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()